In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer

### Step 1: Load the Saved Random Forest Model

In [2]:
import pickle

# Load the saved Random Forest model
with open('D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/rf/random_forest_model.pkl', 'rb') as file:
    rf_model = pickle.load(file)

### Step 2: Prepare Your Test Data

In [3]:
df = pd.read_csv("G:/Data/job_posting/processed/finetune/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')
# replace 'Yes' with True and NaN with False using the fillna() and astype() methods
df['true_ind'] = df['true_ind'].fillna(False).astype(bool)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
# Create a dictionary to map unique soc_codes to sequential integer labels
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}

C:\Users\DELL\AppData\Local\Temp\ipykernel_34836\4103430210.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("G:/Data/job_posting/processed/finetune/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')


In [6]:
# Load datasets from CSV files
train_df = pd.read_csv('G:/Data/job_posting/processed/finetune/train_df_sample.csv', encoding="utf_8_sig")
valid_df = pd.read_csv('G:/Data/job_posting/processed/finetune/valid_df_sample.csv', encoding="utf_8_sig")
test_df = pd.read_csv('G:/Data/job_posting/processed/finetune/test_df_sample.csv', encoding="utf_8_sig")

# drop index column, 'true_ind' and 'sample' columns
train_df = train_df.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code'
train_df['soc_code'] = train_df['soc_code'].astype(str)
train_df['soc_code1'] = train_df['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
test_df = test_df.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the test set
test_df['soc_code'] = test_df['soc_code'].astype(str)
test_df['soc_code1'] = test_df['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
valid_df = valid_df.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the validation set
valid_df['soc_code'] = valid_df['soc_code'].astype(str)
valid_df['soc_code1'] = valid_df['soc_code'].map(soc_code_dict)

# Combine titles and descriptions for tokenization
combined_texts_train = (train_df['工作名称'] + train_df['工作名称'] + " " + train_df['工作描述']).astype(str).tolist()
combined_texts_valid = (valid_df['工作名称'] + valid_df['工作名称'] + " " + valid_df['工作描述']).astype(str).tolist()
combined_texts_test = (test_df['工作名称'] + test_df['工作名称'] + " " + test_df['工作描述']).astype(str).tolist()

#### Text Vectorization with Unigrams and Bigrams:
- Convert the text data into TF-IDF features for the Random Forest model.
- Adjust the TfidfVectorizer to consider both unigram and bigram terms.

In [7]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
combined_texts = combined_texts_train + combined_texts_valid + combined_texts_test
vectorizer.fit(combined_texts)

train_vectors = vectorizer.transform(combined_texts_train)
valid_vectors = vectorizer.transform(combined_texts_valid)
test_vectors = vectorizer.transform(combined_texts_test)

### Step 3: Make Predictions and Evaluate the Model

In [8]:
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

# Make predictions on the test data
predictions = rf_model.predict(test_vectors)

# Assuming 'test_df' is your test dataset and it contains the true labels in 'soc_code1'
true_labels = test_df['soc_code1'].values

# Create a reverse mapping dictionary to convert the numeric labels back to SOC labels
reverse_soc_code_dict = {v: k for k, v in soc_code_dict.items()}

# Convert numeric predictions and true_labels into original SOC codes
soc_predictions = [reverse_soc_code_dict[p] for p in predictions]
soc_true_labels = [reverse_soc_code_dict[l] for l in true_labels]

# Compute classification report
unique_labels = sorted(list(set(soc_true_labels).union(set(soc_predictions))))
report = classification_report(soc_true_labels, soc_predictions, labels=unique_labels, output_dict=True)

# Convert report to a pandas DataFrame
report_df = pd.DataFrame(report).transpose()

# Print the report
print(report_df)

# Save the report to a CSV file
report_df.to_csv('D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/rf/rf_report_df.csv', index=True)

c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score        support
111010         0.176871  0.024976  0.043771    1041.000000
111020         0.244186  0.020772  0.038286    1011.000000
111030         0.695652  0.210526  0.323232      76.000000
112010         0.869811  0.433678  0.578782    1063.000000
112020         0.216117  0.050298  0.081604    1173.000000
...                 ...       ...       ...            ...
537120         0.777778  0.227642  0.352201     246.000000
537190         0.631169  0.615190  0.623077    1185.000000
accuracy       0.367066  0.367066  0.367066       0.367066
macro avg      0.604586  0.304301  0.371714  346379.000000
weighted avg   0.583576  0.367066  0.428814  346379.000000

[409 rows x 4 columns]


c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
